# Zero-Cloud Local Hybrid RAG with SQLite FTS5 & Sentence Transformers

_Authored by: [Çağrı Giray Keşan](https://github.com/Cagrik34)_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/cookbook/blob/main/notebooks/en/zero_cloud_hybrid_rag_sqlite_fts5.ipynb)

---

## 📌 1. Introduction: Solving Vector-Only RAG Bottlenecks

Retrieval-Augmented Generation (RAG) applications typically rely on dense vector databases. While dense embeddings excel at capturing broad semantic context, they frequently suffer from **exact-match blindspots**—failing to reliably retrieve exact numerical identifiers, product codes, or domain-specific acronyms.

Furthermore, deploying standalone vector databases adds operational overhead, cloud latency, and infrastructure cost.

### Key Takeaways of this Recipe:
- **Zero Infrastructure Overhead:** Uses an embedded SQLite database (`:memory:` or local `.db` file) requiring zero external microservices.
- **True Dual-Engine Hybrid Retrieval:** Merges 384-dimensional dense embeddings (`sentence-transformers/all-MiniLM-L6-v2`) with native **SQLite FTS5 BM25** token indexing.
- **Reciprocal Rank Fusion (RRF, $k=60$):** Eliminates score calibration issues between cosine distance and BM25 rank, achieving robust grounded citations (`[1]`, `[2]`).

## 📦 2. Installation & Environment Setup

We only need `sentence-transformers` and `numpy`. SQLite and its FTS5 extension come pre-installed with the Python standard library.

In [ ]:
%pip install -q sentence-transformers numpy

import sqlite3
import numpy as np
from typing import List, Tuple, Dict, Any
from sentence_transformers import SentenceTransformer

# Load lightweight, high-performance open-source embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"✅ Embedding model loaded. Vector dimension: {model.get_sentence_embedding_dimension()}")

## 🏗️ 3. Building the SQLite Dual Hybrid Store

We construct a unified storage schema:
1. `document_chunks`: Stores raw text, document metadata, and float32 binary embedding blobs.
2. `document_chunks_fts`: A virtual full-text index powered by SQLite's native `fts5` module.

In [ ]:
class SQLiteHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                )
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    tokenize='unicode61'
                )
            """)

    def insert_chunk(self, source_file: str, content: str, embedding: np.ndarray) -> None:
        vec = np.array(embedding, dtype=np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm

        with self.conn:
            self.conn.execute(
                "INSERT INTO document_chunks (source_file, content, embedding) VALUES (?, ?, ?)",
                (source_file, content, vec.tobytes())
            )
            self.conn.execute(
                "INSERT INTO document_chunks_fts (content, source_file) VALUES (?, ?)",
                (content, source_file)
            )

    def search_dense(self, query_vec: np.ndarray, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        q_norm = np.linalg.norm(query_vec)
        if q_norm > 0:
            query_vec = query_vec / q_norm

        cursor = self.conn.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        hits = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            sim = float(np.dot(query_vec, doc_vec))
            hits.append((doc_id, src, content, sim))
        hits.sort(key=lambda x: x[3], reverse=True)
        return hits[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        clean_tokens = [t for t in query_text.replace("'", "").replace('"', '').split() if len(t) > 1]
        if not clean_tokens:
            return []
        fts_query = " OR ".join(f'"{t}"' for t in clean_tokens)
        cursor = self.conn.execute(
            "SELECT rowid, source_file, content, rank FROM document_chunks_fts WHERE document_chunks_fts MATCH ? ORDER BY rank LIMIT ?",
            (fts_query, top_k)
        )
        hits = []
        for doc_id, src, content, rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(rank)))
            hits.append((doc_id, src, content, bm25_score))
        return hits

    def hybrid_search(self, query_text: str, query_vec: np.ndarray, top_k: int = 3, rrf_k: int = 60) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_vec, top_k=10)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=10)
        fused_scores = {}
        chunk_map = {}

        for rank, (doc_id, src, content, sim) in enumerate(dense_hits, start=1):
            key = f"{src}::{content[:50]}"
            chunk_map[key] = (src, content, "vector")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (rrf_k + rank))

        for rank, (doc_id, src, content, bm25) in enumerate(sparse_hits, start=1):
            key = f"{src}::{content[:50]}"
            if key not in chunk_map:
                chunk_map[key] = (src, content, "bm25")
            else:
                chunk_map[key] = (src, content, "hybrid")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (rrf_k + rank))

        sorted_keys = sorted(fused_scores.keys(), key=lambda k: fused_scores[k], reverse=True)[:top_k]
        output = []
        for idx, key in enumerate(sorted_keys, start=1):
            src, content, match_type = chunk_map[key]
            output.append({
                "citation_index": idx,
                "source_file": src,
                "content": content,
                "rrf_score": round(fused_scores[key], 4),
                "match_type": match_type
            })
        return output

print("✅ SQLiteHybridRAGStore defined successfully.")

## 📄 4. Ingesting Knowledge Corpus

We index enterprise sample documents containing both descriptive concepts and exact numeric figures.

In [ ]:
store = SQLiteHybridRAGStore()

documents = [
    ("q3_financial_report.pdf", "CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers."),
    ("architecture_specs.md", "Zenith AI leverages Microsoft phi-4-mini (3.8B parameters) for local zero-cloud inference."),
    ("hr_policy_2026.docx", "Remote work expense allowance is capped at 15,000 TL per employee quarterly."),
    ("cluster_ops.md", "Kubernetes cluster autoscaling scales up worker nodes when average CPU utilization exceeds 75% for 3 consecutive minutes.")
]

for src, content in documents:
    emb = model.encode(content)
    store.insert_chunk(src, content, emb)

print(f"✅ Successfully ingested {len(documents)} chunks into SQLite.")

## 🔍 5. Evaluating Retrieval: Vector vs. BM25 vs. Hybrid RRF ($k=60$)

Let's execute a query requiring exact monetary recall.

In [ ]:
query = "What is the quarterly remote work allowance limit in TL?"
query_vec = model.encode(query)

results = store.hybrid_search(query, query_vec, top_k=2)

print(f"Query: '{query}'\n")
for r in results:
    print(f"[{r['citation_index']}] Source: {r['source_file']} | Match: {r['match_type'].upper()} | RRF Score: {r['rrf_score']}")
    print(f"    Content: {r['content']}\n")

## 🤖 6. Synthesizing Grounded Responses with Citations

We format the retrieved passages into grounded context for any open-source or local LLM (e.g. Hugging Face TGI, vLLM, or Transformers pipeline).

In [ ]:
def format_rag_prompt(query: str, retrieved_passages: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for p in retrieved_passages:
        context_blocks.append(f"[{p['citation_index']}] (Source: {p['source_file']}) {p['content']}")
    context_str = "\n\n".join(context_blocks)
    
    return f"""Context information is below:\n---------------------\n{context_str}\n---------------------\nGiven the context above, answer the question: {query}\nStrict rule: Cite the exact source passage using [1], [2] for every factual statement."""

prompt = format_rag_prompt(query, results)
print("📋 Prepared Grounded RAG Prompt:")
print(prompt)
print("\n🤖 Verified Output Simulation:")
print("According to the HR policy documentation [1], the remote work quarterly allowance is strictly capped at 15,000 TL per employee.")